In [13]:

import pandas as pd
import re
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Tải dữ liệu ngôn ngữ
nltk.download('stopwords', quiet=True)
tqdm.pandas(desc="Tiến độ làm sạch LSA")

# Sử dụng các đường dẫn đã thống nhất từ file base
RAW_DIR = "data/raw"
PREPROCESSED_DIR = "data/processed"

In [14]:
# Đọc file đã tải từ bước base
df_corpus = pd.read_parquet(f"../{RAW_DIR}/corpus.parquet")

# Xử lý các giá trị trống
df_corpus['title'] = df_corpus['title'].fillna('')
df_corpus['text'] = df_corpus['text'].fillna('')

print(f"Đã tải {df_corpus.shape[0]:,} bài báo.")

Đã tải 25,657 bài báo.


In [15]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [16]:
port_stemmer = PorterStemmer()
english_stops = set(stopwords.words('english'))

# Tập từ khóa rác học thuật
academic_noise = {
    'paper', 'propos', 'method', 'approach', 'result',
    'show', 'base', 'also', 'howev', 'therefor', 'thu',
    'furthermor', 'present', 'work', 'studi', 'experiment', 'evalu'
}

# Các bộ lọc Regex 
regex_url = re.compile(r'http\S+')
regex_math = re.compile(r'\$.*?\$')
regex_special_chars = re.compile(r'[^a-z0-9\s]')
regex_multi_space = re.compile(r'\s+')

def clean_text_for_lsa(raw_text):
    if not isinstance(raw_text, str):
        return ""
        
    text = raw_text.lower()
    text = regex_url.sub('', text)
    text = regex_math.sub(' math ', text)
    text = regex_special_chars.sub(' ', text)
    text = regex_multi_space.sub(' ', text).strip()
    
    processed_tokens = []
    for word in text.split():
        if len(word) > 2 and word not in english_stops:
            stemmed_word = port_stemmer.stem(word)
            if stemmed_word not in academic_noise:
                processed_tokens.append(stemmed_word)
                
    return ' '.join(processed_tokens)

# Tăng trọng số tiêu đề 
df_corpus['combined_features'] = df_corpus['title'].apply(lambda x: f"{x} " * 5) + df_corpus['text']

print("Bắt đầu chuẩn hóa văn bản cho mô hình LSA...")
df_corpus['text_lsa'] = df_corpus['combined_features'].progress_apply(clean_text_for_lsa)

# Lọc bỏ dòng rỗng sau khi xử lý
df_tv2 = df_corpus.dropna(subset=['text_lsa'])[['_id', 'title', 'text', 'text_lsa']].reset_index(drop=True)

# Lưu file
output_path = f"../{PREPROCESSED_DIR}/data_tv2.parquet"
df_tv2.to_parquet(output_path, index=False)
print(f"Đã lưu dữ liệu LSA tại: {output_path}")

Bắt đầu chuẩn hóa văn bản cho mô hình LSA...


Tiến độ làm sạch LSA:   0%|          | 0/25657 [00:00<?, ?it/s]